In [1]:
import sys, os
sys.path.insert(0, os.getcwd())

import torch
import numpy as np
from model.datasets.scannet_simple import ScanNetSimpleDataset
from model.losses import epipolar_coarse_loss, sampson_epipolar_loss
from gt_epipolar import compute_fundamental_matrix
from train_finetune import get_epipolar_mask_matrix, collate_fn
from config.defaultmf import get_cfg_defaults
from model.lightning_loftr import PL_LoFTR
from model.supervision import compute_supervision

print('Imports OK')

objc[63206]: Class AVFFrameReceiver is implemented in both /Users/siddharthraj/classes/cv/cv_final/myenv/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x10a4c43a8) and /Users/siddharthraj/classes/cv/cv_final/myenv/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x11f6a03a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[63206]: Class AVFAudioReceiver is implemented in both /Users/siddharthraj/classes/cv/cv_final/myenv/lib/python3.12/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x10a4c43f8) and /Users/siddharthraj/classes/cv/cv_final/myenv/lib/python3.12/site-packages/av/.dylibs/libavdevice.62.1.100.dylib (0x11f6a03f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
/Users/siddharthraj/classes/cv/cv_final/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please

Imports OK


In [2]:
# Load dataset
val_ds = ScanNetSimpleDataset(
    '../data/scannet_format/MH_05',
    frame_gap=40, split='test', split_ratio=0.9
)
print(f'Val pairs: {len(val_ds)}')

# Get one batch
batch = collate_fn([val_ds[0]])
print(f'Batch keys: {list(batch.keys())}')
print(f'hw0_c: {batch["hw0_c"]}')
print(f'depth0 max: {batch["depth0"].max().item()}')

  Scenes found: 1 (split=test, ratio=0.9)
    scannet_format: 188 pairs
  Total pairs: 188
Val pairs: 188
Batch keys: ['image0', 'image1', 'depth0', 'K', 'T0', 'T1', 'pair_names', 'hw0_i', 'hw0_c']
hw0_c: (60, 80)
depth0 max: 0.0


In [3]:
# Build model (pretrained, no fine-tuning)
config = get_cfg_defaults()
config.MATCHFORMER.BACKBONE_TYPE = 'litela'
config.MATCHFORMER.SCENS = 'indoor'
config.MATCHFORMER.RESOLUTION = (8, 4)
config.MATCHFORMER.COARSE.D_MODEL = 192
config.MATCHFORMER.COARSE.D_FFN = 192

model = PL_LoFTR(config, pretrained_ckpt='model/weights/indoor-lite-LA.ckpt')
model.eval()
print('Model loaded')

2026-04-08 14:58:34.343 | INFO     | model.lightning_loftr:__init__:54 - Load 'model/weights/indoor-lite-LA.ckpt' as pretrained checkpoint
2026-04-08 14:58:34.344 | INFO     | model.lightning_loftr:__init__:87 - Trainable params: 13,144,320 / 20,256,704 (64.9%) — AttentionBlock3, AttentionBlock4, fine FPN head (BN frozen)


Model loaded


In [4]:
# Step 1: compute supervision (will be empty — no depth)
compute_supervision(batch, config)
print(f'spv_b_ids: {len(batch["spv_b_ids"])} (expected 0 — no depth)')

# Step 2: forward pass
with torch.no_grad():
    model.matcher(batch)

print(f'\nAfter forward pass:')
print(f'  hw0_c: {batch["hw0_c"]}')
print(f'  hw1_c: {batch["hw1_c"]}')
print(f'  conf_matrix: {batch["conf_matrix"].shape}, max: {batch["conf_matrix"].max():.6f}')
print(f'  mkpts0_f: {batch["mkpts0_f"].shape if batch.get("mkpts0_f") is not None else None}')
print(f'  mkpts1_f: {batch["mkpts1_f"].shape if batch.get("mkpts1_f") is not None else None}')
print(f'  m_bids: {batch["m_bids"].shape if batch.get("m_bids") is not None else None}')

spv_b_ids: 0 (expected 0 — no depth)

After forward pass:
  hw0_c: torch.Size([60, 80])
  hw1_c: torch.Size([60, 80])
  conf_matrix: torch.Size([1, 4800, 4800]), max: 0.930524
  mkpts0_f: torch.Size([2935, 2])
  mkpts1_f: torch.Size([2935, 2])
  m_bids: torch.Size([2935])


In [5]:
# Step 3: compute F matrix and epipolar mask
T0 = batch['T0'][0].numpy()
T1 = batch['T1'][0].numpy()
K = batch['K'][0].numpy()
F_mat = compute_fundamental_matrix(T0, T1, K, K)
print(f'F finite: {np.isfinite(F_mat).all()}, norm: {np.linalg.norm(F_mat):.6f}')

H0, W0 = batch['hw0_c']
H1, W1 = batch['hw1_c']
print(f'Coarse dims: ({H0}, {W0}), ({H1}, {W1})')

epi_mask = get_epipolar_mask_matrix(F_mat, H0, W0, H1, W1, tau=24.0)
print(f'epi_mask shape: {epi_mask.shape}')
print(f'epi_mask range: [{epi_mask.min():.4f}, {epi_mask.max():.4f}]')
print(f'Positives (>0.5): {(epi_mask > 0.5).sum().item()} / {epi_mask.numel()}')

F finite: True, norm: 1.092220
Coarse dims: (60, 80), (60, 80)
epi_mask shape: torch.Size([1, 4800, 4800])
epi_mask range: [0.0000, 1.0000]
Positives (>0.5): 2291439 / 23040000


In [6]:
# Step 4: compute SCENES losses
conf_matrix = batch['conf_matrix']

# Coarse loss
loss_coarse = epipolar_coarse_loss(conf_matrix, epi_mask)
print(f'loss_coarse_epi: {loss_coarse.item():.6f}')

# Sampson loss
mkpts0 = batch.get('mkpts0_f')
mkpts1 = batch.get('mkpts1_f')
m_bids = batch.get('m_bids')
if mkpts0 is not None and len(mkpts0) > 0:
    loss_samp = sampson_epipolar_loss(mkpts0, mkpts1, [F_mat], m_bids)
    print(f'loss_sampson: {loss_samp.item():.6f}')
    print(f'num matches: {len(mkpts0)}')
else:
    loss_samp = torch.tensor(0.0)
    print(f'No matches predicted!')

# Total
lambda_epi = 0.5
total = (1 - lambda_epi) * loss_coarse + lambda_epi * loss_samp
print(f'\nTotal SCENES loss: {total.item():.6f}')
print(f'  = {1-lambda_epi} * {loss_coarse.item():.4f} + {lambda_epi} * {loss_samp.item():.4f}')

loss_coarse_epi: 3.980982
loss_sampson: 15.039819
num matches: 2935

Total SCENES loss: 9.510401
  = 0.5 * 3.9810 + 0.5 * 15.0398
